# Simulación del estómago — Fase 1b: validación química con alimentos reales (carne frente a pan)

**Qué cierra esta fase.** En la fase 1a verificamos que *resolvemos bien* nuestras ecuaciones. Aquí preguntamos si *son las ecuaciones correctas*, comparando el modelo con medidas publicadas. Para ello el modelo químico crece en cinco direcciones:

1. **Volumen variable**: el jugo gástrico entra y el contenido se vacía.
2. **Vaciado gástrico** con la regla energética del protocolo INFOGEST semidinámico.
3. **Tampón en equilibrio químico** con varios grupos ionizables, en lugar de la reacción irreversible de 1a.
4. **Almidón y amilasa salival**, que se inactiva al bajar el pH.
5. **Hidrólisis parcial de la proteína**: en el estómago, la pepsina solo rompe una fracción.

**Tres tipos de error, que el cuaderno contabiliza por separado.** Tu idea de comprobar si acoplar modelos reduce el error necesita separar de qué error hablamos:

| Tipo | Pregunta | Cómo se estima aquí | ¿Lo reduce acoplar más física? |
|---|---|---|---|
| **Numérico** | ¿Resuelvo bien mis ecuaciones? | Tolerancias del integrador y muestreo de salida | No: se controla aparte y debe ser despreciable |
| **Paramétrico** | ¿Cuánto cambia la predicción si los parámetros inciertos varían en su rango? | Monte Carlo con hipercubo latino | **A menudo lo aumenta**: más física implica más parámetros |
| **De modelo** (discrepancia) | ¿Cuánto se aleja la predicción de los datos? | Distancia al intervalo medido | **Es el que se espera reducir** |

El cuaderno guarda estos tres errores en un **registro de validación** (`registro_validacion.csv`). Cuando integremos transporte, fluido y partículas, cada versión del modelo añadirá sus filas y podremos comparar versiones con la misma vara de medir. Para empezar, aquí ya comparamos dos versiones: la química de 1a (tampón irreversible) y la de 1b (tampón en equilibrio).

**Calibración, validación y predicción no son lo mismo.** Un dato que se usa para ajustar un parámetro no puede usarse después para validarlo. Cada dato del cuaderno lleva su etiqueta:
- **Calibración**: hidrólisis in vitro del almidón del pan. Fija la cinética de la amilasa.
- **Validación**: retención gástrica de la comida estándar de consenso y pH intragástrico tras una comida.
- **Predicción**: 50 g de carne frente a 50 g de pan. Para este caso no tenemos datos directos, así que se presenta con su banda de incertidumbre.

> ⚠️ Muchos parámetros siguen siendo **supuestos con un rango**, no valores medidos. La tabla de la sección 2 indica el origen de cada uno. El objetivo honesto de esta fase no es "acertar", sino **saber cuánto nos equivocamos y por qué**.

**Uso en Colab**: `Entorno de ejecución → Ejecutar todo`. Tarda unos 3–4 minutos, casi todo en el Monte Carlo. Puedes reducir `N_MC` en la sección 5 para iterar más rápido.

## 1. Ecuaciones: qué cambia respecto a la fase 1a

### 1.1 Balances en cantidades, con volumen variable
Con volumen variable, el balance se escribe en **cantidades** ($N_i$ en mmol o g), no en concentraciones ($c_i = N_i/V$). El volumen sigue

$$ \frac{dV}{dt} = q_s(t) - Q(t), $$

donde $q_s$ es el caudal de jugo gástrico secretado y $Q$ el caudal de vaciado hacia el duodeno. Cada especie cumple

$$ \frac{dN_i}{dt} = \underbrace{s_i(t)}_{\text{secreción}} + \underbrace{V\,\sum_r \nu_{ir}\, r_r}_{\text{reacción}} - \underbrace{Q\,\frac{N_i}{V}}_{\text{vaciado}} . $$

**Hipótesis de vaciado proporcional.** Sale contenido mezclado, con la misma composición que el interior. Es razonable para líquidos, pero **no para sólidos**: el píloro retiene las partículas grandes (la "criba"). Veremos en la sección 6 que esta hipótesis es la principal fuente de error de modelo.

### 1.2 Secreción
Durante el vaciado de la comida, el estómago secreta jugo a un caudal $q_s$ con un aporte de ácido $r_{acid}$ y de pepsinógeno $r_G$. Después, la secreción cae a una fracción basal $f_{basal}$:

$$ s_{H}(t) = r_{acid}\cdot\begin{cases}1 & t < T_e \\ f_{basal} & t \ge T_e\end{cases} \qquad (\text{igual para } q_s \text{ y } r_G). $$

### 1.3 Vaciado: regla energética (INFOGEST semidinámico)
El protocolo semidinámico de INFOGEST fija el tiempo de vaciado a partir del contenido energético de la comida, con un ritmo de referencia de **2 kcal/min**:

$$ T_e = \frac{E_{comida}\ [\mathrm{kcal}]}{2\ \mathrm{kcal/min}}, \qquad Q = q_s + \frac{V_0 - V_{res}}{T_e}\quad (t < T_e). $$

Así el volumen baja linealmente hasta el residuo de ayunas $V_{res}$ en $t = T_e$. El ritmo energético real varía entre personas y comidas; lo tratamos como incierto, entre 1 y 4 kcal/min.

### 1.4 Tampón en equilibrio (relaja la hipótesis H4 de 1a)
La comida contiene grupos ionizables: carboxilos, histidinas, fosfatos… Los modelamos como $n$ familias de ácido débil con $pK_{a,j}$ repartidos entre 2.5 y 6.5. Cada familia cumple la **ley de acción de masas en equilibrio**: $\mathrm{HB}_j \rightleftharpoons \mathrm{H^+} + \mathrm{B}_j^-$, con $K_{a,j} = [\mathrm{H^+}][\mathrm{B}_j^-]/[\mathrm{HB}_j]$.

La variable de estado pasa a ser el **total de protones** $T$ (libres más unidos), que solo cambia por secreción y vaciado. El pH se obtiene resolviendo en cada instante la ecuación algebraica

$$ h + \sum_{j} B_j\,\frac{h}{h + K_{a,j}} = T, \qquad \mathrm{pH} = 3 - \log_{10} h \quad (h \text{ en mM}). $$

**Capacidad tampón.** Por grupos carboxilo, estimamos $\beta \approx 0.3$ mmol de H⁺ por gramo de proteína y por unidad de pH, con un rango de 0.15 a 0.45. Es un orden de magnitud: las proteínas tienen alrededor de 1 mmol/g de Asp+Glu, con $pK_a$ ≈ 4. Con este cambio desaparece la rigidez que encontramos en 1a: ya no hay una reacción rápida $k_b$.

### 1.5 Amilasa salival (almidón → dextrinas)
$$ r_{am} = k_{am}\,\psi(\mathrm{pH})\,c_A, \qquad \psi(\mathrm{pH}) = \frac{1}{1 + 10^{\,n_{am}(\mathrm{pH}_{50} - \mathrm{pH})}} . $$

Usamos $\mathrm{pH}_{50} = 4$ y $n_{am} = 1.7$. Así la actividad cae a la mitad hacia pH 4 y prácticamente desaparece entre pH 3 y 3.5, como se ha medido in vitro. Solo una fracción $f_{acc}$ del almidón es accesible, y otra fracción $f_{oral}$ llega ya hidrolizada desde la boca.

### 1.6 Proteína: hidrólisis gástrica parcial
La proteína se reparte entre una parte hidrolizable por pepsina, $S$ (fracción $f_{pep}$), y una parte que solo se digiere en el intestino, $R$. La cinética de $S$ es la de 1a: Michaelis–Menten con $\phi(\mathrm{pH})$ y activación autocatalítica del pepsinógeno.

### 1.7 Invariantes con entradas y salidas
Para cada especie añadimos su **cantidad vaciada acumulada** $N_i^{out}$, con $dN_i^{out}/dt = Q\,N_i/V$. Las leyes de conservación de 1a pasan a ser balances con entradas y salidas:

$$
\begin{aligned}
& V + V^{out} - \textstyle\int_0^t q_s = V_0 \\
& (S+R+P) + (S+R+P)^{out} = \text{proteína ingerida} \\
& (A + A_r + D) + (\ldots)^{out} = \text{almidón ingerido} \\
& (G + E) + (G+E)^{out} - \textstyle\int_0^t r_G = G_0 + E_0 \\
& T + T^{out} - \textstyle\int_0^t r_{acid} = T_0 \quad(\text{y } B_{tot} + B^{out}_{tot} = B_0) \\
& M + M^{out} = 1 \quad (\text{marcador})
\end{aligned}
$$

## 2. Configuración, modelo y alimentos

In [ ]:
import math, time
from dataclasses import dataclass, field, replace, fields

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from scipy.stats import qmc

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (11, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "lines.linewidth": 2.0, "font.size": 10.5, "legend.frameon": False,
})
GRIS = "#3b3b3b"; GRIS_MEDIO = "#9a9a96"; TINTA2 = "#52514e"
COLOR_COMIDA  = {"carne": "#eb6834", "pan": "#2a78d6"}
COLOR_VERSION = {"1a · tampón irreversible": "#6250d6", "1b · tampón en equilibrio": "#1baf7a"}
COLOR_ERROR   = {"numérico": "#2a78d6", "paramétrico": "#eb6834", "de modelo": "#1baf7a"}

pd.set_option("display.float_format", lambda x: f"{x:.4g}")
VERIFICACION = []

In [ ]:
LN10 = math.log(10.0)

@dataclass
class Param:
    # --- Cinética heredada de la fase 1a ---
    k_a1:   float = 1.7e-3   # 1/(mM·min)    activación del pepsinógeno por ácido
    k_a2:   float = 0.83     # 1/(mM²·min)   activación autocatalítica
    k_cat:  float = 15.0     # (g/L)/(mM·min) proteólisis por pepsina
    K_M:    float = 5.0      # g/L
    pH_opt: float = 2.0
    sigma:  float = 1.5
    k_H:    float = 1e-5     # 1/(mM·min)    hidrólisis ácida directa
    k_b:    float = 1.0      # 1/(mM·min)    solo para el tampón 'irreversible'
    # --- Nuevo: amilasa salival ---
    k_am:   float = 0.4      # 1/min         hidrólisis de almidón accesible a pH alto
    pH50_am: float = 4.0     # —             pH al que la actividad cae a la mitad
    n_am:   float = 1.7      # —             pendiente de la inactivación
    # --- Nuevo: secreción gástrica ---
    r_acid: float = 0.375    # mmol/min      secreción de H+ (22.5 mmol/h)
    q_s:    float = 2.5      # mL/min        caudal de jugo gástrico
    r_G:    float = 1.0e-4   # mmol/min      secreción de pepsinógeno (0.04 mM en el jugo)
    # --- Nuevo: vaciado ---
    vaciado: bool = True
    kcal_min: float = 2.0    # kcal/min      ritmo de vaciado energético
    V_res:  float = 30.0     # mL            volumen residual en ayunas
    f_basal: float = 0.10    # —             secreción tras vaciar la comida, como fracción de la estimulada
    # --- Nuevo: tampón de la comida ---
    tampon: str = "equilibrio"   # 'equilibrio' (1b) o 'irreversible' (1a)
    beta_prot: float = 0.30  # mmol/(g proteína · unidad de pH)
    pKa: tuple = (2.5, 3.5, 4.5, 5.5, 6.5)
    pH_fin_irr: float = 2.0  # pH hasta el que se cuenta el tampón en la versión irreversible


@dataclass
class Comida:
    nombre: str
    masa_g: float            # masa total ingerida (sólidos + bebida), ≈ mL
    proteina_g: float
    almidon_g: float
    kcal: float
    pH0: float = 6.0         # pH de la comida masticada
    f_pep: float = 0.30      # fracción de la proteína hidrolizable por pepsina en el estómago
    f_acc: float = 0.80      # fracción del almidón accesible a la amilasa
    f_oral: float = 0.13     # fracción del almidón ya hidrolizada en la boca
    V_basal: float = 25.0    # mL de jugo en ayunas (pH 1.5, con pepsina activa)


# Índices del vector de estado (estómago); los acumulados vaciados van a continuación
ESTADO = ["V", "X1", "X2", "G", "E", "S", "R", "Pp", "A", "Ar", "D", "M"]
NE = len(ESTADO)
I = {n: i for i, n in enumerate(ESTADO)}


def grupos_tampon(p, comida):
    # mmol de cada grupo ionizable (todos iguales); capacidad total = beta·proteína·(nº de unidades de pH)
    total = p.beta_prot * comida.proteina_g * (max(p.pKa) - min(p.pKa) + 1)
    return np.full(len(p.pKa), total / len(p.pKa))


def protonados(h_mM, Bi_mM, pKa):
    # concentración de grupos protonados (mM) a una concentración libre h (mM)
    return sum(b / (1.0 + 10 ** (3.0 - pk) / h_mM) for b, pk in zip(Bi_mM, pKa))


def resolver_pH(T_mM, Bi_mM, pKa):
    # Resuelve h + sum_i B_i·h/(h+Ka_i) = T para el pH (ley de acción de masas en equilibrio)
    F = lambda x: 10 ** (3.0 - x) + protonados(10 ** (3.0 - x), Bi_mM, pKa) - T_mM
    return brentq(F, -1.0, 9.0, xtol=1e-14, rtol=1e-15)


def psi_amilasa(pH, p):
    return 1.0 / (1.0 + 10 ** (p.n_am * (p.pH50_am - pH)))


def phi_pepsina(pH, p):
    return math.exp(-((pH - p.pH_opt) / p.sigma) ** 2)


def estado_inicial(p, comida):
    y = np.zeros(2 * NE)
    V0 = comida.masa_g + comida.V_basal
    Bi = grupos_tampon(p, comida)                      # mmol
    h_comida = 10 ** (3.0 - comida.pH0)                 # mM
    H_basal = 10 ** (3.0 - 1.5) * comida.V_basal / 1000 # mmol de H+ libre del jugo basal
    if p.tampon == "equilibrio":
        # X1 = protones totales (libres + unidos) ; X2 = grupos tampón totales
        unidos = sum(b / (1 + 10 ** (comida.pH0 - pk)) for b, pk in zip(Bi, p.pKa))
        y[I["X1"]] = h_comida * comida.masa_g / 1000 + unidos + H_basal
        y[I["X2"]] = Bi.sum()
    else:
        # X1 = H+ libre ; X2 = tampón libre que queda por neutralizar hasta pH_fin_irr
        y[I["X1"]] = h_comida * comida.masa_g / 1000 + H_basal
        y[I["X2"]] = sum(b * (1 / (1 + 10 ** (p.pH_fin_irr - pk)) - 1 / (1 + 10 ** (comida.pH0 - pk)))
                         for b, pk in zip(Bi, p.pKa))
    y[I["V"]] = V0
    y[I["E"]] = 0.04 * comida.V_basal / 1000           # pepsina activa del jugo basal (0.04 mM)
    y[I["S"]] = comida.f_pep * comida.proteina_g
    y[I["R"]] = (1 - comida.f_pep) * comida.proteina_g
    y[I["A"]] = (comida.f_acc - comida.f_oral) * comida.almidon_g
    y[I["Ar"]] = (1 - comida.f_acc) * comida.almidon_g
    y[I["D"]] = comida.f_oral * comida.almidon_g
    y[I["M"]] = 1.0                                     # marcador (tipo gammagrafía)
    return y


def pH_del_estado(y, p, comida, Bi_rel=None):
    V = y[I["V"]]
    if p.tampon == "equilibrio":
        T = 1000 * y[I["X1"]] / V
        Bt = 1000 * y[I["X2"]] / V
        n = len(p.pKa)
        return resolver_pH(T, [Bt / n] * n, p.pKa)
    return 3.0 - math.log10(max(1000 * y[I["X1"]] / V, 1e-12))


def make_rhs(p, comida, Q_out, pH_impuesto=None, f_sec=1.0):
    # Q_out: caudal de vaciado (mL/min) constante en el tramo; pH_impuesto(t) para ensayos in vitro
    def f(t, y):
        V = y[I["V"]]
        c = 1000.0 / V                                  # mmol -> mM  y  g -> g/L
        G, E = c * y[I["G"]], c * y[I["E"]]
        S, A = c * y[I["S"]], c * y[I["A"]]
        if pH_impuesto is not None:
            pH = pH_impuesto(t)
        else:
            pH = pH_del_estado(y, p, comida)
        h = 10 ** (3.0 - pH)                            # mM
        r_a = (p.k_a1 * h + p.k_a2 * h * E) * G         # mM/min
        r_e = p.k_cat * E * phi_pepsina(pH, p) * S / (p.K_M + S)   # g/L/min
        r_h = p.k_H * h * S
        r_am = p.k_am * psi_amilasa(pH, p) * A          # g/L/min
        vol = V / 1000.0                                # L
        d = np.zeros(2 * NE)
        d[I["V"]] = f_sec * p.q_s - Q_out
        if p.tampon == "equilibrio" or pH_impuesto is not None:
            d[I["X1"]] = f_sec * p.r_acid
            d[I["X2"]] = 0.0
        else:
            r_b = p.k_b * (c * y[I["X1"]]) * (c * y[I["X2"]])
            d[I["X1"]] = f_sec * p.r_acid - r_b * vol
            d[I["X2"]] = -r_b * vol
        d[I["G"]] = f_sec * p.r_G - r_a * vol
        d[I["E"]] = r_a * vol
        d[I["S"]] = -(r_e + r_h) * vol
        d[I["Pp"]] = (r_e + r_h) * vol
        d[I["A"]] = -r_am * vol
        d[I["D"]] = r_am * vol
        # vaciado proporcional (contenido bien mezclado): sale Q_out·concentración
        sale = Q_out * y[1:NE] / V
        d[1:NE] -= sale
        d[NE] = Q_out                                   # volumen vaciado acumulado
        d[NE + 1:] = sale
        return d
    return f


def tiempo_vaciado(p, comida):
    return comida.kcal / p.kcal_min if p.vaciado else np.inf


def simular(p, comida, t_final=300.0, rtol=1e-7, atol=1e-10, dt_eval=0.25, pH_impuesto=None):
    # Integra por tramos: [0, T_e] vaciando la comida; [T_e, t_final] solo el volumen residual
    y0 = estado_inicial(p, comida)
    T_e = tiempo_vaciado(p, comida)
    V0 = y0[I["V"]]
    tramos = []
    if p.vaciado and T_e < t_final:
        tramos.append((0.0, T_e, p.q_s + (V0 - p.V_res) / T_e, 1.0))
        tramos.append((T_e, t_final, p.f_basal * p.q_s, p.f_basal))
    elif p.vaciado:
        tramos.append((0.0, t_final, p.q_s + (V0 - p.V_res) / T_e, 1.0))
    else:
        tramos.append((0.0, t_final, 0.0, 1.0))
    t_eval_all = np.linspace(0, t_final, int(round(t_final / dt_eval)) + 1)
    ts, ys, y = [], [], y0
    for a, b, Q, fs in tramos:
        te = t_eval_all[(t_eval_all >= a) & (t_eval_all <= b)]
        sol = solve_ivp(make_rhs(p, comida, Q, pH_impuesto, fs), (a, b), y, method="Radau",
                        rtol=rtol, atol=atol, dense_output=True)
        assert sol.success, sol.message
        ts.append(te); ys.append(sol.sol(te).T)
        y = sol.y[:, -1]
    t = np.concatenate(ts); Y = np.vstack(ys)
    t, k = np.unique(t, return_index=True)
    Y = Y[k]
    if pH_impuesto is not None:
        pH = np.array([pH_impuesto(ti) for ti in t])
    else:
        pH = np.array([pH_del_estado(yi, p, comida) for yi in Y])
    return Resultado(t, Y, pH, p, comida)


@dataclass
class Resultado:
    t: np.ndarray
    Y: np.ndarray
    pH: np.ndarray
    p: Param
    comida: Comida

    def col(self, n, salida=False):
        return self.Y[:, I[n] + (NE if salida else 0)]

    def total(self, n):                 # estómago + vaciado acumulado
        return self.col(n) + self.col(n, True)

    # --- Métricas observables ---
    def retencion(self):                # % del marcador en el estómago
        return 100 * self.col("M")

    def t_cruce(self, serie, umbral, bajando=True):
        # primer cruce del umbral, buscando a partir del máximo (o mínimo) de la serie
        i0 = int(np.argmax(serie)) if bajando else int(np.argmin(serie))
        idx = np.where(serie[i0:] < umbral)[0] if bajando else np.where(serie[i0:] > umbral)[0]
        idx = idx + i0
        if idx.size == 0:
            return np.nan
        i = idx[0]
        if i == 0:
            return self.t[0]
        t0, t1, s0, s1 = self.t[i - 1], self.t[i], serie[i - 1], serie[i]
        return t0 + (umbral - s0) * (t1 - t0) / (s1 - s0)

    def frac_proteina_hidrolizada(self):
        return self.total("Pp") / max(self.comida.proteina_g, 1e-12)

    def frac_almidon_hidrolizado(self):
        return self.total("D") / max(self.comida.almidon_g, 1e-12)

    def en(self, serie, t):
        return float(np.interp(t, self.t, serie))

### 2.1 Alimentos y escenarios

La composición se da por cada 100 g y con valores **típicos aproximados** de tablas de composición. Para un alimento concreto conviene sustituirlos por los de BEDCA o USDA FoodData Central. Todas las comidas se toman con agua; la masa total se trata como volumen (densidad ≈ 1).

| Escenario | Composición | Uso |
|---|---|---|
| **Carne** | 50 g de ternera cocinada (≈ 28 % proteína, ≈ 210 kcal/100 g) + 150 mL de agua | Predicción |
| **Pan** | 50 g de pan blanco (≈ 9 % proteína, ≈ 43 % almidón, ≈ 265 kcal/100 g) + 150 mL de agua | Predicción |
| **Comida de consenso** | 120 g de clara de huevo líquida, 2 rebanadas de pan blanco (≈ 45 g), 30 g de mermelada y 120 mL de agua. Total ≈ 255 kcal | Validación del vaciado |
| **Comida de 1000 kcal** | Comida mixta "estándar" sólida y líquida. Su composición exacta no la publica el resumen del estudio; **suponemos** 40 g de proteína, 90 g de almidón y 650 g en total | Validación del pH |
| **Pan in vitro** | Solo almidón, con el pH impuesto: rampa de 6 a 2 en 30 min ("snack") o en 60 min ("comida") | Calibración de la amilasa |

In [ ]:
CARNE    = Comida("carne",    masa_g=50 + 150, proteina_g=0.28 * 50, almidon_g=0.0,       kcal=2.10 * 50)
PAN      = Comida("pan",      masa_g=50 + 150, proteina_g=0.09 * 50, almidon_g=0.43 * 50, kcal=2.65 * 50)
CONSENSO = Comida("consenso", masa_g=120 + 45 + 30 + 120, proteina_g=13 + 4, almidon_g=19, kcal=255)
MIL_KCAL = Comida("1000 kcal", masa_g=650, proteina_g=40, almidon_g=90, kcal=1000)

p0 = Param()
filas = []
for c in (CARNE, PAN, CONSENSO, MIL_KCAL):
    filas.append({"escenario": c.nombre, "masa total [g]": c.masa_g, "proteína [g]": c.proteina_g,
                  "almidón [g]": c.almidon_g, "energía [kcal]": c.kcal,
                  "tiempo de vaciado T_e [min]": tiempo_vaciado(p0, c),
                  "capacidad tampón [mmol/pH]": p0.beta_prot * c.proteina_g})
pd.DataFrame(filas).style.format(precision=1).hide(axis="index")

### 2.2 Parámetros: valor nominal, rango y origen
Los rangos de la última columna son los que recorre el Monte Carlo de la sección 5.

In [ ]:
ORIGEN = {
    "k_a1": ("1/(mM·min)", "heredado de 1a (ilustrativo)", "fijo"),
    "k_a2": ("1/(mM²·min)", "heredado de 1a (ilustrativo)", "fijo"),
    "k_cat": ("(g/L)/(mM·min)", "heredado de 1a (ilustrativo)", "5 – 45 (log)"),
    "K_M": ("g/L", "heredado de 1a (ilustrativo)", "fijo"),
    "pH_opt": ("—", "óptimo típico de pepsina", "fijo"),
    "sigma": ("—", "ilustrativo", "fijo"),
    "k_H": ("1/(mM·min)", "ilustrativo, despreciable", "fijo"),
    "k_b": ("1/(mM·min)", "solo versión 1a", "fijo"),
    "k_am": ("1/min", "CALIBRADO en la sección 4", "k_min – 4·k_min (log)"),
    "pH50_am": ("—", "amilasa a mitad de actividad ~pH 4 (in vitro)", "fijo"),
    "n_am": ("—", "inhibición completa a pH 3–3.5 (in vitro)", "fijo"),
    "r_acid": ("mmol/min", "supuesto: 22.5 mmol/h", "0.19 – 0.75 (log)"),
    "q_s": ("mL/min", "supuesto", "1.5 – 4 (log)"),
    "r_G": ("mmol/min", "supuesto: 0.04 mM en el jugo", "fijo"),
    "vaciado": ("—", "", ""),
    "kcal_min": ("kcal/min", "INFOGEST semidinámico: 2 kcal/min", "1 – 4 (log)"),
    "V_res": ("mL", "supuesto", "fijo"),
    "f_basal": ("—", "supuesto", "fijo"),
    "tampon": ("—", "versión del modelo", ""),
    "beta_prot": ("mmol/(g·pH)", "estimado por grupos carboxilo", "0.15 – 0.45"),
    "pKa": ("—", "repartidos en el rango de titulación", "fijo"),
    "pH_fin_irr": ("—", "solo versión 1a", "fijo"),
}
EXTRA = [("f_pep (comida)", 0.30, "—", "supuesto: fracción hidrolizable por pepsina", "0.10 – 0.50"),
         ("pH0 (comida)", 6.0, "—", "pH de la comida masticada (supuesto)", "5.5 – 6.5"),
         ("f_acc (pan)", 0.80, "—", "≈20 % del almidón queda sin digerir in vitro", "fijo"),
         ("f_oral (pan)", 0.13, "—", "≥13 % hidrolizado en la fase oral in vitro", "fijo")]
tabla = [{"parámetro": f.name, "valor": str(getattr(p0, f.name)), "unidades": ORIGEN[f.name][0],
          "origen": ORIGEN[f.name][1], "rango Monte Carlo": ORIGEN[f.name][2]}
         for f in fields(p0) if f.name not in ("vaciado", "tampon")]
tabla += [{"parámetro": a, "valor": str(b), "unidades": c, "origen": d, "rango Monte Carlo": e} for a, b, c, d, e in EXTRA]
pd.DataFrame(tabla).style.hide(axis="index")

## 3. Verificación de las partes nuevas

Antes de comparar con datos, comprobamos que las ecuaciones nuevas están bien resueltas. Usamos el integrador implícito `Radau` con tolerancias estrictas y lo contrastamos con soluciones exactas.

### T5 — Volumen y marcador con secreción y vaciado constantes
Con $q_s$ y $Q$ constantes: $V(t) = V_0 + (q_s - Q)\,t$. Para el marcador, $\dot M = -Q M/V$, que da
$$ M(t) = \left(\frac{V(t)}{V_0}\right)^{-Q/(q_s - Q)} . $$

In [ ]:
def test_T5(rtol=1e-11):
    p = replace(Param(), r_acid=0.0, r_G=0.0)
    c = Comida("tracer", masa_g=300, proteina_g=1, almidon_g=0, kcal=0, V_basal=0)
    Q = 6.0
    y0 = estado_inicial(p, c)
    f = make_rhs(p, c, Q, pH_impuesto=lambda t: 6.0)
    t = np.linspace(0, 40, 81)
    sol = solve_ivp(f, (0, 40), y0, method="Radau", rtol=rtol, atol=1e-14, t_eval=t)
    V_ex = y0[I["V"]] + (p.q_s - Q) * t
    M_ex = (V_ex / y0[I["V"]]) ** (-Q / (p.q_s - Q))
    eV = np.max(np.abs(sol.y[I["V"]] - V_ex)) / y0[I["V"]]
    eM = np.max(np.abs(sol.y[I["M"]] - M_ex))
    return t, sol, V_ex, M_ex, eV, eM

t5, sol5, V_ex, M_ex, eV, eM = test_T5()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.plot(t5, V_ex, color=GRIS, lw=3, alpha=0.4, label="exacta"); a1.plot(t5, sol5.y[I["V"]], "o", ms=4, mfc="none", color="#1baf7a", label="Radau")
a1.set(xlabel="t [min]", ylabel="V [mL]", title=f"Volumen  (error rel. {eV:.1e})"); a1.legend()
a2.plot(t5, M_ex, color=GRIS, lw=3, alpha=0.4, label="exacta"); a2.plot(t5, sol5.y[I["M"]], "o", ms=4, mfc="none", color="#1baf7a", label="Radau")
a2.set(xlabel="t [min]", ylabel="marcador M", title=f"Marcador  (error abs. {eM:.1e})"); a2.legend()
fig.suptitle("T5 · Secreción + vaciado constantes", fontweight="bold"); fig.tight_layout(); plt.show()
VERIFICACION.append({"test": "T5 volumen y marcador (exacta)", "error": max(eV, eM), "tolerancia": 1e-8})

### T6 — Amilasa con rampa de pH (solución exacta)
Con el pH impuesto $\mathrm{pH}(t) = \mathrm{pH}_i - \Delta\, t/T_r$, la ecuación es lineal: $\dot A = -k_{am}\,\psi(\mathrm{pH}(t))\,A$. Su solución es
$$ A(t) = A_0 \exp\!\Big(-k_{am}\int_0^t \psi\,ds\Big), \qquad \int \psi\, d\mathrm{pH} = \mathrm{pH} + \frac{\ln\!\big(1 + 10^{\,n(\mathrm{pH}_{50}-\mathrm{pH})}\big)}{n\ln 10}, $$
y el cambio de variable $ds = -(T_r/\Delta)\,d\mathrm{pH}$ da la integral en el tiempo de forma cerrada.

In [ ]:
def rampa_pH(T_r, pH_i=6.0, pH_f=2.0):
    return lambda t: pH_i - (pH_i - pH_f) * min(t, T_r) / T_r

def integral_psi(t, T_r, p, pH_i=6.0, pH_f=2.0):
    D = pH_i - pH_f
    F = lambda x: x + np.log1p(10 ** (p.n_am * (p.pH50_am - x))) / (p.n_am * LN10)
    tt = np.minimum(t, T_r)
    I_ramp = (T_r / D) * (F(pH_i) - F(pH_i - D * tt / T_r))
    return I_ramp + psi_amilasa(pH_f, p) * np.maximum(t - T_r, 0)

def escenario_in_vitro(k_am):
    p = replace(Param(), k_am=k_am, vaciado=False, q_s=0.0, r_acid=0.0, r_G=0.0)
    c = Comida("pan in vitro", masa_g=100, proteina_g=0, almidon_g=10, kcal=0, V_basal=0)
    return p, c

p6, c6 = escenario_in_vitro(0.3)
r6 = simular(p6, c6, t_final=90, rtol=1e-11, atol=1e-14, dt_eval=0.5, pH_impuesto=rampa_pH(30))
A_ex = estado_inicial(p6, c6)[I["A"]] * np.exp(-p6.k_am * integral_psi(r6.t, 30, p6))
eA = np.max(np.abs(r6.col("A") - A_ex)) / A_ex[0]
print(f"T6: error relativo máximo de A(t) = {eA:.2e}")
VERIFICACION.append({"test": "T6 amilasa con rampa de pH (exacta)", "error": eA, "tolerancia": 1e-8})

### T7 — Solucionador del pH en equilibrio
Con un solo grupo tampón, la ecuación $h + B\,h/(h+K_a) = T$ es cuadrática y tiene solución cerrada:
$$ h = \tfrac12\Big[-(K_a + B - T) + \sqrt{(K_a + B - T)^2 + 4\,T\,K_a}\Big] = \frac{2\,T K_a}{(K_a + B - T) + \sqrt{(K_a + B - T)^2 + 4\,T\,K_a}} . $$
Para la referencia usamos la segunda forma. La primera, aunque es matemáticamente idéntica, pierde cifras por **cancelación** cuando $T$ es pequeño: es un ejemplo de error numérico que viene de la fórmula exacta, no del solucionador. Comparamos `resolver_pH` con ella en todo el rango de acidez.

In [ ]:
Ka, Bt = 10 ** (3 - 4.0), 40.0
Ts = np.logspace(-3, 2.5, 200)
b_ = Ka + Bt - Ts
h_ex = 2 * Ts * Ka / (b_ + np.sqrt(b_ ** 2 + 4 * Ts * Ka))   # forma estable (evita cancelación)
pH_num = np.array([resolver_pH(T, [Bt], (4.0,)) for T in Ts])
e_pH = np.max(np.abs(pH_num - (3 - np.log10(h_ex))))
print(f"T7: error máximo en pH = {e_pH:.2e}")
VERIFICACION.append({"test": "T7 solucionador de pH (1 grupo, exacta)", "error": e_pH, "tolerancia": 1e-9})

### T8 — Balances con secreción y vaciado
Se simulan las cuatro comidas con las dos versiones del tampón y se mide el mayor residuo relativo de los invariantes de 1.7. Recuerda lo aprendido en 1a: un residuo pequeño **no prueba precisión**, pero un residuo grande **sí prueba un error de programación**.

In [ ]:
def integral_secrecion(t, tasa, p, T_e):
    return tasa * np.minimum(t, T_e) + p.f_basal * tasa * np.maximum(t - T_e, 0)

def residuos_balance(r):
    p, c, t = r.p, r.comida, r.t
    y0 = estado_inicial(p, c)
    T_e = tiempo_vaciado(p, c)
    res = {}
    res["volumen"] = np.abs(r.total("V") - integral_secrecion(t, p.q_s, p, T_e) - y0[I["V"]]) / y0[I["V"]]
    prot0, alm0 = max(c.proteina_g, 1e-12), max(c.almidon_g, 1e-12)
    res["proteína"] = np.abs(r.total("S") + r.total("R") + r.total("Pp") - c.proteina_g) / prot0
    res["almidón"] = np.abs(r.total("A") + r.total("Ar") + r.total("D") - c.almidon_g) / alm0
    G_in = integral_secrecion(t, p.r_G, p, T_e)
    res["enzima"] = np.abs(r.total("G") + r.total("E") - G_in - y0[I["G"]] - y0[I["E"]]) / (G_in[-1] + y0[I["E"]])
    A_in = integral_secrecion(t, p.r_acid, p, T_e)
    if p.tampon == "equilibrio":
        res["protones"] = np.abs(r.total("X1") - A_in - y0[I["X1"]]) / (A_in[-1] + y0[I["X1"]])
        res["tampón"] = np.abs(r.total("X2") - y0[I["X2"]]) / y0[I["X2"]]
    else:
        res["protones"] = np.abs(r.total("X1") + (y0[I["X2"]] - r.total("X2")) - A_in - y0[I["X1"]]) / (A_in[-1] + y0[I["X1"]])
    res["marcador"] = np.abs(r.total("M") - 1.0)
    return {k: float(v.max()) for k, v in res.items()}

filas = []
for version in ("irreversible", "equilibrio"):
    for c in (CARNE, PAN, CONSENSO, MIL_KCAL):
        r = simular(replace(Param(), tampon=version), c, t_final=360, rtol=1e-11, atol=1e-14)
        filas.append({"versión": version, "comida": c.nombre, **residuos_balance(r)})
df_bal = pd.DataFrame(filas)
max_bal = df_bal.drop(columns=["versión", "comida"]).max().max()
VERIFICACION.append({"test": "T8 balances con entradas/salidas", "error": max_bal, "tolerancia": 1e-8})
df_bal.style.format("{:.1e}", subset=df_bal.columns[2:], na_rep="—").hide(axis="index")

### T9 — La versión 1b contiene a la 1a (test de regresión)
Si se apagan el vaciado y el caudal de jugo (la secreción aporta ácido sin volumen), se elige el tampón irreversible y se quitan el almidón y el jugo basal, el modelo 1b debe reproducir **exactamente** el modelo 1a. Este test protege contra romper lo que ya estaba verificado al añadir física nueva.

In [ ]:
def rhs_1a(p, r_s):
    # Lado derecho de la fase 1a, en concentraciones (copiado de aquel cuaderno)
    def f(t, y):
        H, B, G, E, S, P = y
        pH = 3.0 - math.log10(max(H, 1e-12))
        phi = phi_pepsina(pH, p)
        r_b = p.k_b * H * B
        r_a = (p.k_a1 * H + p.k_a2 * H * E) * G
        r_e = p.k_cat * E * phi * S / (p.K_M + S)
        r_h = p.k_H * H * S
        return np.array([r_s - r_b, -r_b, -r_a, r_a, -r_e - r_h, r_e + r_h])
    return f

V_1a = 800.0                                         # mL
p9 = replace(Param(), tampon="irreversible", vaciado=False, q_s=0.0, r_G=0.0, r_acid=0.35 * V_1a / 1000)
c9 = Comida("1a", masa_g=V_1a, proteina_g=0, almidon_g=0, kcal=0, V_basal=0)
y0 = np.zeros(2 * NE)
y0[I["V"]] = V_1a
for nombre, conc in (("X1", 0.01), ("X2", 45.0), ("G", 0.02), ("S", 30.0)):
    y0[I[nombre]] = conc * V_1a / 1000
t9 = np.linspace(0, 240, 961)
sol_b = solve_ivp(make_rhs(p9, c9, 0.0), (0, 240), y0, method="Radau", rtol=1e-11, atol=1e-14, t_eval=t9)
sol_a = solve_ivp(rhs_1a(p9, 0.35), (0, 240), [0.01, 45.0, 0.02, 0.0, 30.0, 0.0], method="Radau",
                  rtol=1e-11, atol=1e-14, t_eval=t9)
conc_b = sol_b.y[[I["X1"], I["X2"], I["G"], I["E"], I["S"], I["Pp"]]] * 1000 / V_1a
esc = np.max(np.abs(sol_a.y), axis=1, keepdims=True)
e9 = np.max(np.abs(conc_b - sol_a.y) / esc)
print(f"T9: diferencia relativa máxima 1b frente a 1a = {e9:.2e}")
VERIFICACION.append({"test": "T9 regresión: 1b reproduce 1a", "error": e9, "tolerancia": 1e-7})

### T10 — Error numérico de las métricas
Lo que comparamos con datos no son curvas, sino **métricas**: tiempos de cruce, retenciones, fracciones hidrolizadas. Estas métricas tienen dos fuentes de error numérico: la tolerancia del integrador y el muestreo de la salida, porque el tiempo de cruce se interpola entre muestras. Las estimamos comparando la configuración de producción (`rtol=1e-7`, muestreo cada 0.25 min) con una de referencia (`rtol=1e-11`, cada 0.05 min).

In [ ]:
def metricas(r):
    ret = r.retencion()
    return {
        "retención 1 h [%]": r.en(ret, 60), "retención 2 h [%]": r.en(ret, 120), "retención 4 h [%]": r.en(ret, 240),
        "t hasta pH ≤ 2 [min]": r.t_cruce(r.pH, 2.0), "t hasta pH ≤ 3 [min]": r.t_cruce(r.pH, 3.0),
        "pH máximo": float(r.pH.max()),
        "T50 vaciado [min]": r.t_cruce(ret, 50.0), "T90 vaciado [min]": r.t_cruce(ret, 10.0),
        "proteína hidrolizada [%]": 100 * float(r.frac_proteina_hidrolizada()[-1]),
        "almidón hidrolizado [%]": 100 * float(r.frac_almidon_hidrolizado()[-1]),
    }

PROD = dict(rtol=1e-7, atol=1e-10, dt_eval=0.25)
REF  = dict(rtol=1e-11, atol=1e-14, dt_eval=0.05)
T_FIN = {"carne": 240, "pan": 240, "consenso": 300, "1000 kcal": 360}

def error_numerico(p, c):
    m_p = metricas(simular(p, c, t_final=T_FIN[c.nombre], **PROD))
    m_r = metricas(simular(p, c, t_final=T_FIN[c.nombre], **REF))
    return m_p, {k: abs(m_p[k] - m_r[k]) for k in m_p}

filas = []
ENUM = {}
for version in ("irreversible", "equilibrio"):
    for c in (CARNE, PAN, CONSENSO, MIL_KCAL):
        m, e = error_numerico(replace(Param(), tampon=version), c)
        ENUM[(version, c.nombre)] = e
        filas.append({"versión": version, "comida": c.nombre, **{k: e[k] for k in e}})
df_enum = pd.DataFrame(filas)
peor = np.nanmax(df_enum[["t hasta pH ≤ 2 [min]", "T50 vaciado [min]", "retención 2 h [%]"]].to_numpy())
VERIFICACION.append({"test": "T10 error numérico de métricas < 0.05 (min o %)", "error": peor, "tolerancia": 0.05})
df_enum.style.format("{:.1e}", subset=df_enum.columns[2:], na_rep="—").hide(axis="index")

## 4. Calibración: cinética de la amilasa con el pan in vitro

**Dato.** En digestiones in vitro de pan con amilasa salival:
- Cerca del 80 % del almidón se hidroliza en los primeros 20–30 min de la fase gástrica, y la curva se estanca justo por debajo de ese valor.
- Las curvas son prácticamente iguales tanto si el pH baja de 6 a 2 en 30 min como si lo hace en 60 min.

Fuente: Freitas et al., *Food & Function* 2018, enlazado al final.

**Cómo lo convertimos en criterios numéricos** (esta traducción es nuestra y es discutible):
1. Hidrólisis a los 20 min con la rampa de 30 min: **≥ 75 %**.
2. Diferencia entre las rampas de 60 y 30 min al final: **≤ 5 puntos**.

El segundo criterio es el interesante. Solo se cumple si la amilasa es lo bastante rápida como para agotar el almidón accesible *antes* de que el ácido la inactive. Si fuera lenta, una acidificación más lenta le daría más tiempo y digeriría más. Así que el dato **acota $k_{am}$ por abajo**, pero no por arriba.

In [ ]:
k_grid = np.logspace(-2, 0.5, 40)
h20, delta = [], []
for k in k_grid:
    p_, c_ = escenario_in_vitro(k)
    r30 = simular(p_, c_, t_final=120, dt_eval=0.5, pH_impuesto=rampa_pH(30))
    r60 = simular(p_, c_, t_final=120, dt_eval=0.5, pH_impuesto=rampa_pH(60))
    h20.append(100 * r30.en(r30.frac_almidon_hidrolizado(), 20))
    delta.append(100 * (r60.frac_almidon_hidrolizado()[-1] - r30.frac_almidon_hidrolizado()[-1]))
h20, delta = np.array(h20), np.array(delta)
factible = (h20 >= 75) & (np.abs(delta) <= 5)
K_MIN = float(k_grid[factible].min())
K_NOM = 2 * K_MIN
print(f"k_am mínimo compatible con los datos: {K_MIN:.3f} 1/min  ->  nominal 2·k_min = {K_NOM:.3f} 1/min")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
for k, ls in ((0.5 * K_MIN, ":"), (K_NOM, "-")):
    for T_r, col in ((30, "#2a78d6"), (60, "#eb6834")):
        p_, c_ = escenario_in_vitro(k)
        r = simular(p_, c_, t_final=120, dt_eval=0.5, pH_impuesto=rampa_pH(T_r))
        a1.plot(r.t, 100 * r.frac_almidon_hidrolizado(), color=col, ls=ls, lw=1.8,
                label=f"rampa {T_r} min, k_am = {k:.2f}")
a1.axhspan(75, 80, color=GRIS_MEDIO, alpha=0.2, lw=0)
a1.text(118, 84, "dato: ~80 %", ha="right", va="center", fontsize=9, color=TINTA2)
a1.set(xlabel="t [min]", ylabel="almidón hidrolizado [%]", ylim=(0, 90), title="Curvas in vitro (continua: calibrada)")
a1.legend(fontsize=8.5, loc="lower right")

a2.semilogx(k_grid, h20, color="#2a78d6", label="hidrólisis a 20 min [%] (≥ 75)")
a2.semilogx(k_grid, np.abs(delta) * 10, color="#eb6834", label="|rampa 60 − rampa 30| × 10 [pts] (≤ 50)")
a2.axvspan(K_MIN, k_grid.max(), color="#1baf7a", alpha=0.12, lw=0)
a2.text(K_MIN * 1.3, 120, "región compatible\ncon los datos", fontsize=9, color=TINTA2)
a2.set(xlabel="k_am [1/min]", ylabel="valor del criterio", title="Restricciones sobre k_am")
a2.legend(fontsize=8.5, loc="upper right")
fig.suptitle("Calibración de la amilasa salival (pan in vitro)", fontweight="bold"); fig.tight_layout(); plt.show()

## 5. Predicción: 50 g de carne frente a 50 g de pan

La simulación nominal se acompaña de una **banda de incertidumbre paramétrica** (percentiles 5–95). Para obtenerla, un Monte Carlo con hipercubo latino recorre los rangos de la tabla 2.2.

Definimos el "tiempo de digestión gástrica" con métricas observables:
- **T50 / T90**: tiempo hasta vaciar el 50 % / 90 % del marcador.
- **t hasta pH ≤ 3**: aproximadamente cuando la amilasa queda inactiva y la pepsina empieza a trabajar.
- **Proteína / almidón hidrolizado**: porcentaje del total ingerido que se hidroliza antes de salir del estómago.

In [ ]:
N_MC = 40          # nº de muestras de Monte Carlo (40 ≈ 3 min en Colab; súbelo para bandas más suaves)
SEMILLA = 7
RANGOS = [          # (nombre, mínimo, máximo, escala, ¿pertenece a la comida?)
    ("beta_prot", 0.15, 0.45, "lin", False),
    ("r_acid",    0.19, 0.75, "log", False),
    ("q_s",       1.5,  4.0,  "log", False),
    ("kcal_min",  1.0,  4.0,  "log", False),
    ("k_cat",     5.0,  45.0, "log", False),
    ("k_am",      K_MIN, 4 * K_MIN, "log", False),
    ("f_pep",     0.10, 0.50, "lin", True),
    ("pH0",       5.5,  6.5,  "lin", True),
]

def muestras_lhs(n, semilla=SEMILLA):
    u = qmc.LatinHypercube(d=len(RANGOS), seed=semilla).random(n)
    out = []
    for fila in u:
        d = {}
        for (nombre, lo, hi, escala, _), x in zip(RANGOS, fila):
            d[nombre] = lo * (hi / lo) ** x if escala == "log" else lo + (hi - lo) * x
        out.append(d)
    return out

def aplicar(p, c, d):
    de_comida = {nombre for nombre, *_, es_comida in RANGOS if es_comida}
    dp = {k: v for k, v in d.items() if k not in de_comida}
    dc = {k: v for k, v in d.items() if k in de_comida}
    return replace(p, **dp), replace(c, **dc)

def monte_carlo(p_base, c_base, n=N_MC, curvas=False):
    filas, series = [], []
    for d in muestras_lhs(n):
        p, c = aplicar(p_base, c_base, d)
        r = simular(p, c, t_final=T_FIN[c_base.nombre], **PROD)
        filas.append(metricas(r))
        if curvas:
            series.append(np.vstack([r.pH, r.retencion(), 100 * r.frac_proteina_hidrolizada(),
                                     100 * r.frac_almidon_hidrolizado()]))
    return pd.DataFrame(filas), (r.t, np.array(series)) if curvas else None

P_NOM = replace(Param(), k_am=K_NOM)
t0 = time.time()
NOMINAL, MC, CURVAS = {}, {}, {}
for c in (CARNE, PAN):
    NOMINAL[c.nombre] = simular(P_NOM, c, t_final=T_FIN[c.nombre], **PROD)
    MC[("equilibrio", c.nombre)], CURVAS[c.nombre] = monte_carlo(P_NOM, c, curvas=True)
print(f"Monte Carlo carne + pan: {time.time() - t0:.0f} s")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True)
paneles = [("pH", "pH", 0), ("retención del marcador [%]", "Vaciado gástrico", 1),
           ("proteína hidrolizada [%]", "Proteína hidrolizada en el estómago", 2),
           ("almidón hidrolizado [%]", "Almidón hidrolizado (incluye fase oral)", 3)]
for a, (ylab, tit, k) in zip(ax.flat, paneles):
    for nombre in ("carne", "pan"):
        if k == 3 and nombre == "carne":
            continue
        t, S = CURVAS[nombre]
        lo, hi = np.percentile(S[:, k, :], [5, 95], axis=0)
        a.fill_between(t, lo, hi, color=COLOR_COMIDA[nombre], alpha=0.18, lw=0)
        r = NOMINAL[nombre]
        y = [r.pH, r.retencion(), 100 * r.frac_proteina_hidrolizada(), 100 * r.frac_almidon_hidrolizado()][k]
        a.plot(r.t, y, color=COLOR_COMIDA[nombre], label=f"50 g de {nombre}")
    a.set(ylabel=ylab, title=tit)
    if k == 0:
        a.axhline(3, color=GRIS_MEDIO, lw=0.8, ls=":"); a.text(235, 3.1, "pH 3", ha="right", fontsize=9, color=TINTA2)
ax[0, 0].legend(); ax[1, 0].set_xlabel("t [min]"); ax[1, 1].set_xlabel("t [min]")
fig.suptitle("Predicción: 50 g de carne frente a 50 g de pan (línea: nominal · banda: percentiles 5–95)", fontweight="bold")
fig.tight_layout(); plt.show()

claves = ["T50 vaciado [min]", "T90 vaciado [min]", "t hasta pH ≤ 3 [min]", "proteína hidrolizada [%]", "almidón hidrolizado [%]"]
filas = []
for nombre in ("carne", "pan"):
    m_nom = metricas(NOMINAL[nombre])
    df = MC[("equilibrio", nombre)]
    for k in claves:
        if nombre == "carne" and k.startswith("almidón"):
            continue
        filas.append({"comida": nombre, "métrica": k, "nominal": m_nom[k],
                      "p5": np.nanpercentile(df[k], 5), "p95": np.nanpercentile(df[k], 95)})
PRED = pd.DataFrame(filas)
PRED.style.format({"nominal": "{:.1f}", "p5": "{:.1f}", "p95": "{:.1f}"}).hide(axis="index")

**Lectura.**
- **Vaciado.** La carne tiene menos energía en 50 g que el pan, así que con la regla energética se vacía *antes*. Esto contradice la intuición clínica de que la carne "pesa" más. Además, los T50 de 20–25 min están muy por debajo de los tiempos clínicos de vaciado de sólidos: Malmud et al. dan un T½ de 70–140 min para comidas estándar. La causa es que el modelo todavía no sabe que la carne es un sólido difícil de romper: la regla energética solo mira las kcal, y el vaciado proporcional trata todo como líquido. La validación de la sección 6 confirma que este es el error de modelo dominante. Lo resolverá la fase de partículas, con criba pilórica y rotura mecánica.
- **pH en ayunas.** Tras vaciarse, el pH baja a ~0.9, que es el del propio jugo secretado. El estómago real en ayunas está a pH 1.3–1.5 porque se mezcla con saliva y moco. No afecta a las métricas, que se deciden antes, pero es otra simplificación pendiente.
- **pH.** La carne tampona mucho más, porque tiene unas tres veces más proteína. El pH tarda más en bajar de 3 y la pepsina se activa tarde, cuando buena parte de la carne ya ha salido.
- **Proteína hidrolizada.** En el estómago se hidroliza poca proteína, y la banda de incertidumbre es muy ancha. Es coherente con el papel secundario de la digestión gástrica de proteínas, pero el valor depende casi por completo de $f_{pep}$, que **no hemos podido acotar con datos**.
- **Almidón del pan.** Se hidroliza menos que en la calibración in vitro, porque in vivo el pH de esta comida pequeña baja más deprisa que en las rampas del ensayo.

## 6. Validación frente a datos y comparación de versiones

| Dato | Valor medido | Comida | Fuente |
|---|---|---|---|
| Retención a 1 h | 37–90 % | Consenso (clara + pan + mermelada) | Consenso SNMMI/ANMS, rangos normales |
| Retención a 2 h | 30–60 % | ídem | ídem |
| Retención a 4 h | 0–10 % | ídem | ídem |
| Tiempo hasta pH ≈ 2 | ≤ 240 min | 1000 kcal, mixta | Russell et al. 1993 (79 adultos sanos) |
| pH máximo tras la comida | 5.8–6.7 (RIC; mediana 6.2) | ídem | ídem |

Cada métrica se evalúa con las dos versiones del tampón, cada una con su Monte Carlo. Así respondemos a una pregunta concreta: **¿añadir el equilibrio ácido–base reduce el error de modelo?**

In [ ]:
OBJETIVOS = [  # (comida, métrica, mínimo, máximo)
    (CONSENSO, "retención 1 h [%]", 37, 90),
    (CONSENSO, "retención 2 h [%]", 30, 60),
    (CONSENSO, "retención 4 h [%]", 0, 10),
    (MIL_KCAL, "t hasta pH ≤ 2 [min]", 0, 240),
    (MIL_KCAL, "pH máximo", 5.8, 6.7),
]
VERSIONES = {"1a · tampón irreversible": "irreversible", "1b · tampón en equilibrio": "equilibrio"}

t0 = time.time()
for etiqueta, version in VERSIONES.items():
    for c in (CONSENSO, MIL_KCAL):
        if (version, c.nombre) not in MC:
            MC[(version, c.nombre)], _ = monte_carlo(replace(P_NOM, tampon=version), c)
print(f"Monte Carlo de validación: {time.time() - t0:.0f} s")

def discrepancia(x, lo, hi):
    return 0.0 if lo <= x <= hi else min(abs(x - lo), abs(x - hi))

REGISTRO = []
for etiqueta, version in VERSIONES.items():
    p_v = replace(P_NOM, tampon=version)
    for c, met, lo, hi in OBJETIVOS:
        m_nom, e_num = error_numerico(p_v, c)
        df = MC[(version, c.nombre)]
        p5, p95 = np.nanpercentile(df[met], [5, 95])
        REGISTRO.append({
            "versión": etiqueta, "comida": c.nombre, "métrica": met,
            "nominal": m_nom[met], "p5": p5, "p95": p95, "objetivo_min": lo, "objetivo_max": hi,
            "error_numérico": e_num[met], "error_paramétrico": (p95 - p5) / 2,
            "error_de_modelo": discrepancia(m_nom[met], lo, hi),
            "banda_solapa_dato": bool(p95 >= lo and p5 <= hi),
        })
REG = pd.DataFrame(REGISTRO)
REG.style.format({c: "{:.3g}" for c in ["nominal", "p5", "p95", "error_numérico", "error_paramétrico", "error_de_modelo"]}).hide(axis="index")

In [ ]:
mets = [o[1] for o in OBJETIVOS]
fig, axs = plt.subplots(1, len(mets), figsize=(13, 4.0))
for a, (c, met, lo, hi) in zip(axs, OBJETIVOS):
    a.axhspan(lo, hi, color=GRIS_MEDIO, alpha=0.25, lw=0)
    for j, etiqueta in enumerate(VERSIONES):
        fila = REG[(REG["versión"] == etiqueta) & (REG["métrica"] == met)].iloc[0]
        x = j
        a.errorbar(x, fila["nominal"], yerr=[[fila["nominal"] - fila["p5"]], [fila["p95"] - fila["nominal"]]],
                   fmt="o", ms=7, capsize=4, lw=1.6, color=COLOR_VERSION[etiqueta])
    a.set_xticks([0, 1]); a.set_xticklabels(["1a", "1b"]); a.set_xlim(-0.6, 1.6)
    a.set_title(met.replace(" [", "\n["), fontsize=10)
    a.grid(axis="x", visible=False)
axs[0].set_ylabel("valor de la métrica")
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
fig.legend(handles=[Patch(color=GRIS_MEDIO, alpha=0.35, label="intervalo medido")] +
           [Line2D([], [], marker="o", ls="none", color=col, label=lab) for lab, col in COLOR_VERSION.items()],
           loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Validación: nominal (punto) y percentiles 5–95 (barra) frente al dato (banda gris)", y=1.08, fontweight="bold")
fig.tight_layout(); plt.show()

**Qué dicen los datos sobre el modelo.**
- **pH máximo.** La versión 1a (tampón irreversible) queda **fuera** del intervalo medido. La 1b (equilibrio) cae **dentro**. Aquí añadir física reduce el error de modelo, que es justo lo que querías comprobar. El motivo es que, con tampón irreversible, el ácido basal del estómago en ayunas se queda libre al principio. Con equilibrio, la comida lo absorbe y el pH arranca cerca del de la comida.
- **Tiempo hasta pH 2.** Las dos versiones dan el mismo valor, y es por construcción: la versión 1a cuenta como tampón exactamente lo que la 1b neutraliza hasta pH 2. Ambas cumplen el dato, que es poco exigente ("antes de 4 h").
- **Retención a 1 h.** El valor nominal queda algo por debajo del rango normal, pero la banda paramétrica lo alcanza. Este fallo podría deberse a los parámetros, por ejemplo a un ritmo de vaciado más lento que 2 kcal/min.
- **Retención a 2 h.** Las dos versiones fallan por mucho. Es un error **estructural**: ni siquiera el extremo de la banda paramétrica llega al dato, así que ningún ajuste de parámetros lo arregla. La gammagrafía marca el **sólido** (la clara de huevo), que el píloro retiene hasta que se tritura a menos de 1–2 mm. Nuestro vaciado proporcional trata todo como un líquido mezclado. Es la mejor justificación cuantitativa para la fase de **partículas con criba pilórica**.

## 7. Presupuesto de error y registro de validación

Para comparar errores de métricas con unidades distintas, dividimos cada uno por la anchura del intervalo medido. Un error de 1 equivale a "tan grande como todo el rango normal".

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
ancho = 0.26
CAMPO = {"numérico": "error_numérico", "paramétrico": "error_paramétrico", "de modelo": "error_de_modelo"}
xs, etiquetas_x = [], []
for i, (c, met, lo, hi) in enumerate(OBJETIVOS):
    centro = []
    for j, etiqueta in enumerate(VERSIONES):
        fila = REG[(REG["versión"] == etiqueta) & (REG["métrica"] == met)].iloc[0]
        x0 = i * 3.0 + j
        for k, (tipo, col) in enumerate(COLOR_ERROR.items()):
            val = fila[CAMPO[tipo]] / (hi - lo)
            if val > 0:                       # error nulo: sin barra
                ax.bar(x0 + (k - 1) * ancho, val, width=ancho * 0.92, color=col,
                       label=tipo if (i == 0 and j == 0) else None)
        xs.append(x0); etiquetas_x.append(etiqueta[:2]); centro.append(x0)
    ax.text(np.mean(centro), 5, met.split(" [")[0], ha="center", fontsize=9, color=TINTA2)
ax.set_yscale("log"); ax.set_ylim(1e-14, 30)
ax.axhline(1, color=GRIS_MEDIO, lw=0.8, ls="--")
ax.set_xticks(xs); ax.set_xticklabels(etiquetas_x)
ax.set_ylabel("error / anchura del intervalo medido")
ax.set_title("Presupuesto de error por métrica y versión (sin barra = error nulo; línea discontinua = anchura del intervalo)")
ax.legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, -0.1)); ax.grid(axis="x", visible=False)
fig.tight_layout(); plt.show()

resumen_v = REG.groupby("versión").agg(
    metricas_dentro=("error_de_modelo", lambda s: int((s == 0).sum())),
    bandas_que_solapan=("banda_solapa_dato", "sum"),
    error_modelo_normalizado_medio=("error_de_modelo",
        lambda s: float(np.mean(s.to_numpy() / np.array([hi - lo for _, _, lo, hi in OBJETIVOS])))))
display(resumen_v)

In [ ]:
# Registro de validación: cada fase futura añadirá sus filas con su etiqueta de versión
import os
GUARDAR_EN_DRIVE = False          # True para montar Google Drive en Colab y guardar allí
ruta = "registro_validacion.csv"
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/simulacion_estomago", exist_ok=True)
    ruta = "/content/drive/MyDrive/simulacion_estomago/registro_validacion.csv"
reg_guardar = REG.assign(fase="1b", fecha=pd.Timestamp.now().strftime("%Y-%m-%d"))
if os.path.exists(ruta):
    previo = pd.read_csv(ruta)
    previo = previo[~previo["versión"].isin(reg_guardar["versión"])]   # sustituye las filas de estas versiones
    reg_guardar = pd.concat([previo, reg_guardar], ignore_index=True)
reg_guardar.to_csv(ruta, index=False)
print(f"Registro guardado en {ruta}  ({len(reg_guardar)} filas)")

## 8. Resumen de verificación

In [ ]:
ver = pd.DataFrame(VERIFICACION)
ver["resultado"] = np.where(ver["error"] < ver["tolerancia"], "✅ pasa", "❌ FALLA")
display(ver.style.format({"error": "{:.1e}", "tolerancia": "{:.0e}"}).hide(axis="index"))
n_ok = int((ver["error"] < ver["tolerancia"]).sum())
print(f"\n{n_ok}/{len(ver)} tests superados")
assert n_ok == len(ver), "Hay tests de verificación que no pasan"

## 9. Conclusiones y siguientes pasos

**Sobre tu pregunta: ¿acoplar modelos reduce el error?** En esta fase ya aparece la respuesta matizada:
- Añadir el **equilibrio ácido–base** reduce el error de modelo en el pH inicial. Hace pasar la métrica de fuera a dentro del intervalo medido, sin coste en las demás. En el registro, el error de modelo normalizado medio baja de ≈0.37 a ≈0.21.
- El error de modelo **dominante**, el vaciado de sólidos, no lo reduce ninguna mejora química. Solo lo puede reducir acoplar la **física de partículas**. El presupuesto de error indica qué módulo merece la pena integrar primero.
- El error **paramétrico** es grande: la anchura de la banda es comparable a la de los intervalos medidos. Cada acoplamiento nuevo añadirá parámetros. Para que el modelo integrado gane precisión neta, harán falta datos que los acoten.
- El error **numérico** está al menos tres órdenes de magnitud por debajo de los otros dos, así que no limita nada.

**Datos pendientes, que acotarían los parámetros más influyentes**
1. Grado de hidrólisis gástrica de la carne cocinada in vitro (INFOGEST), para acotar $f_{pep}$ y $k_{cat}$.
2. Curvas de titulación de carne y pan, para medir $\beta$ directamente.
3. Gammagrafía o resonancia de comidas de carne frente a pan, para validar la predicción de la sección 5.

**Siguiente fase (2): transporte 1D** con advección, difusión y reacción. Esta química entra como término de reacción y sus resultados irán al mismo registro de validación.

### Referencias de los datos
- Abell T.L. et al. *Consensus recommendations for gastric emptying scintigraphy* (SNMMI/ANMS), resumido en J Nucl Med Technol 36(1):44: <https://tech.snmjournals.org/content/36/1/44>
- Malmud L.S. et al., valores normales de vaciado de sólidos: <https://numebook.uchile.cl/artmes/NMDigitalBook/GastrointestinalExaminations_Web/SGES_NormalValues_Malmud.htm>
- Russell T.L. et al. *Upper gastrointestinal pH in seventy-nine healthy, elderly, North American men and women*, Pharm Res (1993): <https://link.springer.com/article/10.1023/A:1018970323716>
- Freitas D., Le Feunteun S. et al. *The important role of salivary α-amylase in the gastric digestion of wheat bread starch*, Food & Function (2018): <https://pubs.rsc.org/en/content/articlehtml/2018/fo/c7fo01484h>
- Mulet-Cabero A.-I. et al. *A standardised semi-dynamic in vitro digestion method suitable for food – an international consensus*, Food & Function (2020): <https://pubs.rsc.org/en/content/articlehtml/2020/fo/c9fo01293a>